# RepoSage: Local Codebase Intelligence & Debugging Copilot

**Kushagra - Resume RAG Project**

RepoSage is a RAG system over a software repository: source code, architecture notes, troubleshooting notes, and logs. This notebook intentionally follows the syntax and learning flow from **Sir's In-Class Labs 1-6** as closely as possible.

### Class-lab reuse
| Lab | RepoSage use |
|---|---|
| Lab 1 - LLM Generation Parameters | low-temperature factual generation |
| Lab 2 - Introduction to LangChain | `init_chat_model`, prompts and runnable chains |
| Lab 3 - Structured Output Generation | Pydantic schema + `with_structured_output` |
| Lab 4 - RAG | load -> split -> embed -> Chroma -> BM25/EnsembleRetriever |
| Lab 5 - Advanced RAG | MMR + MultiQuery experiment |
| Lab 6 - RAG with LangGraph | `START -> retrieve -> generate -> END` |

**Small project-specific addition:** Python AST creates function/class retrieval units and source metadata.


## 1. Initial Setup


In [1]:
# Run once if needed:
# !pip install -r requirements.txt
# !ollama pull qwen3:8b
# !ollama pull nomic-embed-text

import os, json, warnings
import pandas as pd
from pathlib import Path
from uuid import uuid4
from dotenv import load_dotenv
from IPython.display import display, Markdown

warnings.filterwarnings('ignore')
load_dotenv(override=True)

NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == 'notebooks' else NOTEBOOK_DIR
DEMO_REPO = PROJECT_ROOT / 'dataset' / 'knowledge_base' / 'demo_repo'
EVAL_FILE = PROJECT_ROOT / 'evaluation' / 'evaluation_questions.csv'
CHROMA_DIR = PROJECT_ROOT / 'chroma_db'
print('Project root:', PROJECT_ROOT)


Project root: C:\Users\Eldoria\Pictures\portfolio projects\RepoSage_End_to_End_Sir_Code


### 1.1 Chat model - follows Lab 2 / Lab 4


In [2]:
from langchain.chat_models import init_chat_model

model_name = os.getenv('OLLAMA_CHAT_MODEL', 'qwen3:8b')
llm = init_chat_model(
    model_name,
    model_provider='ollama',
    temperature=0.1,  # Lab 1 idea: low randomness for factual answers
)
llm


ChatOllama(model='qwen3:8b', temperature=0.1)

### 1.2 Embeddings - follows Lab 4 / Lab 5


In [3]:
from langchain_ollama import OllamaEmbeddings

embeddings_model = OllamaEmbeddings(model=os.getenv('OLLAMA_EMBED_MODEL', 'nomic-embed-text'))
embeddings_model


OllamaEmbeddings(model='nomic-embed-text', validate_model_on_init=False, base_url=None, client_kwargs={}, async_client_kwargs={}, sync_client_kwargs={}, mirostat=None, mirostat_eta=None, mirostat_tau=None, num_ctx=None, num_gpu=None, keep_alive=None, num_thread=None, repeat_last_n=None, repeat_penalty=None, temperature=None, stop=None, tfs_z=None, top_k=None, top_p=None)

## 2. Custom Dataset: Source Code + Docs + Logs


In [4]:
from repo_loader import load_repository

all_docs = load_repository(DEMO_REPO)
print('Loaded document units:', len(all_docs))

preview = pd.DataFrame([
    {
        'path': d.metadata.get('relative_path'),
        'symbol': d.metadata.get('symbol'),
        'symbol_type': d.metadata.get('symbol_type'),
        'type': d.metadata.get('document_type'),
    }
    for d in all_docs
])
display(preview.head(20))


Loaded document units: 28


,path,symbol,symbol_type,type
0,config/settings.yaml,NaN,NaN,documentation
1,docs/api_reference.md,NaN,NaN,documentation
2,docs/architecture.md,NaN,NaN,documentation
3,docs/deployment.md,NaN,NaN,documentation
4,docs/troubleshooting.md,NaN,NaN,documentation
5,incidents/incident_001.md,NaN,NaN,documentation
6,logs/app.log,NaN,NaN,documentation
7,logs/deployment.log,NaN,NaN,documentation
8,src/api.py,api.py,file,code
9,src/api.py,get_profile,FunctionDef,code


### RepoSage-specific loader addition
Python files are still documents, but the loader also uses the standard-library `ast` module to create focused documents for classes/functions. This is the only major code-aware addition; the RAG pipeline below stays close to class.


## 3. Split Documents - same Lab 4 baseline


In [5]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    add_start_index=True,
)
all_splits = text_splitter.split_documents(all_docs)
print(f'Split {len(all_docs)} document units into {len(all_splits)} chunks.')


Split 28 document units into 30 chunks.


## 4. Store Chunks in Chroma - follows Lab 4 / Lab 5


In [6]:
from langchain_chroma import Chroma

vector_store_chroma = Chroma(
    collection_name='reposage-rag',
    embedding_function=embeddings_model,
    persist_directory=str(CHROMA_DIR),
)

# For a clean notebook demo, clear the collection before adding current chunks.
try:
    vector_store_chroma.reset_collection()
except Exception:
    pass

uuids = [str(uuid4()) for _ in range(len(all_splits))]
vector_store_chroma.add_documents(documents=all_splits, ids=uuids)
print('Indexed chunks:', len(all_splits))


Indexed chunks: 30


## 5. Retrieval Methods from Class


### 5.1 Similarity retrieval


In [7]:
similarity_retriever = vector_store_chroma.as_retriever(
    search_type='similarity',
    search_kwargs={'k': 4}
)


### 5.2 MMR retrieval - Advanced RAG experiment


In [8]:
mmr_retriever = vector_store_chroma.as_retriever(
    search_type='mmr',
    search_kwargs={'k': 4}
)


### 5.3 Hybrid retrieval: BM25 + Chroma - follows Lab 4


In [9]:
from langchain.retrievers import BM25Retriever, EnsembleRetriever

bm25_retriever = BM25Retriever.from_documents(documents=all_splits)
bm25_retriever.k = 4

chroma_retriever = vector_store_chroma.as_retriever(search_kwargs={'k': 4})

ensemble_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, chroma_retriever],
    weights=[0.7, 0.3],  # same weighting idea as class
)


### 5.4 Inspect retrieval before generation


In [10]:
question = 'How is token expiry configured?'
retrieved_docs = ensemble_retriever.invoke(question)

for i, doc in enumerate(retrieved_docs, 1):
    print(f'--- Result {i} ---')
    print('Path:', doc.metadata.get('relative_path'))
    print('Symbol:', doc.metadata.get('symbol'))
    print(doc.page_content[:600], '\n')


--- Result 1 ---
Path: docs/troubleshooting.md
Symbol: None
# Troubleshooting Runbook

## Database timeout

Symptoms include `ConnectionTimeoutError`, requests hanging during database access, or messages saying a connection could not be acquired from the pool.

Check in this order:

1. Verify `DATABASE_URL` points to the intended host/database.
2. Verify DNS/network reachability from the application environment.
3. Check whether all pooled connections are busy. The default `DB_POOL_SIZE` is 5.
4. Compare the failure with application logs before changing credentials.

## JWT expiry complaints

If users are logged out after roughly 30 minutes, check `ACC 

--- Result 2 ---
Path: docs/api_reference.md
Symbol: None
# API Reference

## GET /profile

Requires `Authorization: Bearer <token>`. The request is validated by `validate_bearer_header`, decoded by the authentication layer, and then the profile handler loads the user record.

## GET /health

Returns a lightweight service-health respon

## 6. Prompt - follows Lab 4 ChatPromptTemplate


In [11]:
from langchain_core.prompts import ChatPromptTemplate

system_message_template = """You are RepoSage, a local codebase intelligence and debugging assistant.

Use ONLY the retrieved repository context supplied to you.
Do not invent files, functions, settings, log events, bugs or runtime behavior.
Separate facts directly visible in the repository from diagnostic possibilities.
Give safe debugging/navigation steps and cite relevant paths/functions.
If evidence is insufficient, set information_gap to true.

Retrieved context:
{context}
"""

reposage_prompt = ChatPromptTemplate([
    ('system', system_message_template),
    ('human', 'Developer question: {question}'),
])


## 7. Structured Output - follows Lab 3


In [12]:
from pydantic import BaseModel, Field

class RepoSageResponse(BaseModel):
    answer: str = Field(description='Grounded answer to the codebase/debugging question')
    observed_evidence: list[str] = Field(description='Facts directly supported by retrieved evidence')
    likely_causes: list[str] = Field(description='Possible causes, not confirmed facts')
    recommended_steps: list[str] = Field(description='Safe debugging/navigation steps')
    source_references: list[str] = Field(description='Files/functions used')
    information_gap: bool = Field(description='True when retrieved evidence is insufficient')

structured_llm = llm.with_structured_output(RepoSageResponse)


## 8. LCEL RAG Chain - same idea as Lab 4


In [13]:
def format_docs(docs):
    formatted = []
    for doc in docs:
        formatted.append(
            f"PATH: {doc.metadata.get('relative_path')}\n"
            f"SYMBOL: {doc.metadata.get('symbol')}\n"
            f"CONTENT: {doc.page_content}"
        )
    return '\n\n'.join(formatted)


def ask_reposage_lcel(question):
    docs = ensemble_retriever.invoke(question)
    context = format_docs(docs)
    chain = reposage_prompt | structured_llm
    answer = chain.invoke({'question': question, 'context': context})
    return answer, docs


In [14]:
answer, evidence = ask_reposage_lcel('How is token expiry configured?')
print(answer.model_dump())


{'answer': "Token expiry is configured via the `ACCESS_TOKEN_EXPIRE_MINUTES` setting. Here's how it's implemented:\n\n1. **Default value**: 30 minutes (defined in `src/config.py`)\n2. **Configuration source**: `config/settings.yaml` under `security.access_token_expire_minutes`\n3. **Code usage**: The value is used in `src/auth.py:create_access_token()` to calculate the JWT expiration time (`exp` claim)\n4. **Override method**: Change via environment variable (not hard-coded in `auth.py`)\n\nSee: `docs/troubleshooting.md` for guidance on adjusting this setting.", 'observed_evidence': ['docs/troubleshooting.md mentions ACCESS_TOKEN_EXPIRE_MINUTES default is 30', 'docs/architecture.md states expiration comes from ACCESS_TOKEN_EXPIRE_MINUTES', 'src/auth.py uses ACCESS_TOKEN_EXPIRE_MINUTES to calculate expiration', 'config/settings.yaml contains security.access_token_expire_minutes: 30'], 'likely_causes': ['If tokens expire too quickly, check if environment variable override is correctly se

## 9. Advanced RAG Experiment - MultiQuery from Lab 5


In [15]:
from langchain.retrievers.multi_query import MultiQueryRetriever

base_retriever = vector_store_chroma.as_retriever(
    search_type='similarity',
    search_kwargs={'k': 2}
)

mq_retriever = MultiQueryRetriever.from_llm(
    retriever=base_retriever,
    llm=llm,
    include_original=True,
)

# Example:
# mq_docs = mq_retriever.invoke('Why is the DB hanging?')
# [(d.metadata.get('relative_path'), d.metadata.get('symbol')) for d in mq_docs]


## 10. Final Workflow with LangGraph - follows Lab 6


In [16]:
from typing_extensions import TypedDict
from typing import List
from langchain_core.documents import Document
from langgraph.graph import START, StateGraph, END

class State(TypedDict):
    question: str
    context: List[Document]
    answer: RepoSageResponse


def retrieve(state: State):
    retrieved_docs = ensemble_retriever.invoke(state['question'])
    return {'context': retrieved_docs}


def generate(state: State):
    docs_content = format_docs(state['context'])
    response = (reposage_prompt | structured_llm).invoke({
        'question': state['question'],
        'context': docs_content,
    })
    return {'answer': response}


graph_builder = StateGraph(State)
graph_builder.add_node('retrieve', retrieve)
graph_builder.add_node('generate', generate)
graph_builder.add_edge(START, 'retrieve')
graph_builder.add_edge('retrieve', 'generate')
graph_builder.add_edge('generate', END)
graph = graph_builder.compile()


C:\Users\Eldoria\Pictures\portfolio projects\RepoSage_End_to_End_Sir_Code\.venv\Lib\site-packages\langgraph\cache\base\__init__.py:8: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


In [17]:
result = graph.invoke({'question': 'Why could the database connection be timing out?'})
print(result['answer'].model_dump())


{'answer': 'The database connection could be timing out due to:\n\n1. **Connection Pool Exhaustion** (most likely)\n   - Default `DB_POOL_SIZE` is 5 connections\n   - Incident 001 shows that exceeding this limit during load tests caused timeouts\n   - Verify if application concurrency exceeds database capacity\n\n2. **Network/DNS Issues**\n   - Check `DATABASE_URL` validity and DNS resolution\n   - Ensure application environment can reach the database host\n\n3. **Database Host Unreachability**\n   - Network latency or firewall issues\n   - Verify database service is running and accepting connections\n\n4. **Configuration Mismatch**\n   - Compare pool size/settings with target database limits\n   - Ensure credentials are correct (though this typically causes auth errors, not timeouts)\n\n**Recommended Steps**: Check the order in docs/troubleshooting.md, verify DATABASE_URL, test network connectivity, and monitor pool usage metrics. The incident_001.md example shows how excessive concur

## 11. Retrieval Evaluation - Similarity vs MMR vs Hybrid


In [18]:
eval_df = pd.read_csv(EVAL_FILE)
print('Evaluation questions:', len(eval_df))
display(eval_df.head())


Evaluation questions: 20


,question_id,question,expected_source,expected_symbol
0,Q01,How is token expiry configured?,src/config.py,NaN
1,Q02,Where are access-token expiration claims created?,src/auth.py,create_access_token
2,Q03,What algorithm is associated with access tokens?,src/config.py,NaN
3,Q04,Why could a database connection time out?,logs/app.log,NaN
4,Q05,What is the default database pool size?,src/config.py,NaN


In [19]:
def source_hit(docs, expected_source):
    paths = [d.metadata.get('relative_path', '') for d in docs]
    return int(any(p == expected_source or p.endswith(expected_source) for p in paths))


def evaluate_retriever(method='hybrid'):
    rows = []
    retriever_map = {
        'similarity': similarity_retriever,
        'mmr': mmr_retriever,
        'hybrid': ensemble_retriever,
    }
    retriever = retriever_map[method]

    for _, row in eval_df.iterrows():
        docs = retriever.invoke(row['question'])
        rows.append({
            'question_id': row['question_id'],
            'method': method,
            'source_hit': source_hit(docs, row['expected_source']),
            'retrieved_paths': ' | '.join(d.metadata.get('relative_path','') for d in docs),
        })
    return pd.DataFrame(rows)

# Run when ready:
# similarity_results = evaluate_retriever('similarity')
# mmr_results = evaluate_retriever('mmr')
# hybrid_results = evaluate_retriever('hybrid')
# all_results = pd.concat([similarity_results, mmr_results, hybrid_results])
# display(all_results.groupby('method')[['source_hit']].mean().round(3))


## 12. Controlled Failure Test


In [20]:
# RepoSage should admit when the repository does not contain the answer.
# unsupported = graph.invoke({'question': 'What cloud region is production deployed in?'})
# print(unsupported['answer'].model_dump())


## 13. Demo Helper


In [21]:
def ask_reposage(question):
    result = graph.invoke({'question': question})
    answer = result['answer']
    display(Markdown('### RepoSage Answer'))
    display(Markdown(answer.answer))
    display(Markdown('**Observed evidence**'))
    for item in answer.observed_evidence:
        display(Markdown(f'- {item}'))
    if answer.likely_causes:
        display(Markdown('**Likely causes**'))
        for item in answer.likely_causes:
            display(Markdown(f'- {item}'))
    display(Markdown('**Recommended steps**'))
    for item in answer.recommended_steps:
        display(Markdown(f'- {item}'))
    display(Markdown('**Sources**'))
    for item in answer.source_references:
        display(Markdown(f'- {item}'))
    return answer, result['context']

# ask_reposage('Explain the authentication flow.')


## 14. Limitations

1. Static code cannot prove production runtime behavior.
2. RepoSage only knows files included in the index.
3. Large monorepos may need language-aware chunking and metadata filters beyond this course baseline.
4. Secret files are intentionally excluded.
5. The 1000/200 chunking strategy is retained mainly for consistency with Sir's Lab 4 and should be benchmarked before production use.
